## PART 1 of 3 — Optuna Hyperparameter Tuning + Cross-Validation

> **Outputs saved to:** `./outputs/models/` — upload to Kaggle Dataset before running Part 2

# Alzheimer's Disease Detection — Full Research Pipeline
## Dual-Channel ConvNeXt-Tiny + Swin-V2-T + CBAM | FedBN Federated Learning

Architecture : ConvNeXt-Tiny (CNN) + Swin-V2-T (Transformer) + CBAM
Loss         : WCE(label_smoothing=0.1) + Focal + Supervised Contrastive Loss
Tuning       : Optuna TPE sampler + MedianPruner (20 trials)
Validation   : 5-Fold Patient-Level Stratified Cross-Validation
Federated    : FedAvg | FedProx | FedBN
XAI          : Grad-CAM++ | Score-CAM | SHAP | LIME | Attention Rollout
Analysis     : Ablation (13 variants) | t-SNE/UMAP | Calibration |
               Uncertainty (MC-Dropout) | Statistical Tests | Robustness |
               Fairness | Error Analysis | Model Saving

Datasets:
  Client 1 (Primary)  : ninadaithal/imagesoasis  (OASIS-1, ~80 K slices, patient IDs)
  Client 2 (External) : uraninjo /original/       (6 400 images, external test)



## Step 1 -- Clinical Task Definition & Label Mapping

Fulfils **Proposal Step 1**: define the classification task and document
how every dataset label maps to the unified label set.
All downstream code uses `CLASS2IDX` -- never raw string labels.


In [1]:
# ====================================================================
# STEP 1 -- CLINICAL TASK DEFINITION & LABEL MAPPING
# Proposal requirement: define ONE label dictionary; keep it identical
# across all splits, datasets, and evaluation stages.
# ====================================================================

import textwrap

# 1a. Unified class set (4-class multi-stage task)
CLASSES     = ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']
CLASS2IDX   = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS   = {i: c for c, i in CLASS2IDX.items()}
NUM_CLASSES = len(CLASSES)

# 1b. Clinical stage descriptions
CLINICAL_DESC = {
    'NonDemented':       'Cognitively normal; CDR = 0',
    'VeryMildDemented':  'Very mild cognitive impairment; CDR = 0.5',
    'MildDemented':      'Mild dementia; CDR = 1',
    'ModerateDemented':  'Moderate dementia; CDR = 2',
}

# 1c. Dataset-specific label harmonisation maps
LABEL_MAP_OASIS = {
    'Non Demented':       'NonDemented',
    'NonDemented':        'NonDemented',
    'Very Mild Demented': 'VeryMildDemented',
    'VeryMildDemented':   'VeryMildDemented',
    'Mild Demented':      'MildDemented',
    'MildDemented':       'MildDemented',
    'Moderate Demented':  'ModerateDemented',
    'ModerateDemented':   'ModerateDemented',
}

LABEL_MAP_EXTERNAL = {
    'NonDemented':       'NonDemented',
    'VeryMildDemented':  'VeryMildDemented',
    'MildDemented':      'MildDemented',
    'ModerateDemented':  'ModerateDemented',
}

# 1d. Task description
TASK_NAME        = "4-class Alzheimer's disease multi-stage classification"
TASK_DESCRIPTION = textwrap.dedent("""
    Task  : Multi-stage AD detection (4 classes, Proposal Step 1).
    Labels: NonDemented | VeryMildDemented | MildDemented | ModerateDemented
    Metric: AUC-macro (primary), F1-macro, Balanced Accuracy (secondary).
    Split : Patient-level stratified split -- zero data leakage.
    Data  : OASIS (internal train/val/test) + External (external test).
""").strip()

# 1e. Print label registry
print('=' * 65)
print(f'  TASK: {TASK_NAME}')
print('=' * 65)
print(f'{"Index":<8} {"Class Name":<22} {"Clinical Description"}')
print('-' * 65)
for cls, idx in CLASS2IDX.items():
    print(f'  {idx:<6} {cls:<22} {CLINICAL_DESC[cls]}')
print()
print('OASIS label mapping:')
for src_lbl, tgt_lbl in LABEL_MAP_OASIS.items():
    if src_lbl != tgt_lbl:
        print(f"  '{src_lbl}'  ->  '{tgt_lbl}'  (idx={CLASS2IDX[tgt_lbl]})")
print()
print('External dataset label mapping:')
for src_lbl, tgt_lbl in LABEL_MAP_EXTERNAL.items():
    print(f"  '{src_lbl}'  ->  idx={CLASS2IDX[tgt_lbl]}")
print()
print(TASK_DESCRIPTION)


  TASK: 4-class Alzheimer's disease multi-stage classification
Index    Class Name             Clinical Description
-----------------------------------------------------------------
  0      NonDemented            Cognitively normal; CDR = 0
  1      VeryMildDemented       Very mild cognitive impairment; CDR = 0.5
  2      MildDemented           Mild dementia; CDR = 1
  3      ModerateDemented       Moderate dementia; CDR = 2

OASIS label mapping:
  'Non Demented'  ->  'NonDemented'  (idx=0)
  'Very Mild Demented'  ->  'VeryMildDemented'  (idx=1)
  'Mild Demented'  ->  'MildDemented'  (idx=2)
  'Moderate Demented'  ->  'ModerateDemented'  (idx=3)

External dataset label mapping:
  'NonDemented'  ->  idx=0
  'VeryMildDemented'  ->  idx=1
  'MildDemented'  ->  idx=2
  'ModerateDemented'  ->  idx=3

Task  : Multi-stage AD detection (4 classes, Proposal Step 1).
Labels: NonDemented | VeryMildDemented | MildDemented | ModerateDemented
Metric: AUC-macro (primary), F1-macro, Balanced Accuracy

## Section 1 — Imports & Global Configuration

In [2]:
import os, re, time
KAGGLE_START_TIME = time.time()
MAX_KAGGLE_SECONDS = 11.3 * 3600  # 11 hours 18 mins
import random, copy, json, time, warnings
import copy   
import json   
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, fbeta_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, brier_score_loss,
    precision_score, recall_score, matthews_corrcoef,
)
from sklearn.manifold import TSNE
from sklearn.calibration import calibration_curve
from scipy.stats import wilcoxon

import timm
from torchvision import transforms

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")


SEED         = 42
NUM_CLASSES  = 4
CLASSES      = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
CLASS2IDX    = {c: i for i, c in enumerate(CLASSES)}


DATA_ROOT     = "/kaggle/input/datasets"                                              
OASIS_ROOT    = f"{DATA_ROOT}/ninadaithal/imagesoasis/Data"
EXTERNAL_ROOT = f"{DATA_ROOT}/uraninjo/augmented-alzheimer-mri-dataset/OriginalDataset"


OASIS_FOLDER_MAP = {
    "Non Demented"       : "NonDemented",
    "Mild Dementia"      : "MildDemented",
    "Moderate Dementia"  : "ModerateDemented",
    "Very mild Dementia" : "VeryMildDemented",
}

IMG_SIZE       = 256
BATCH_SIZE      = 32         # safe for kaggle P100/T4 OOM         # larger batch -> smoother gradients
NUM_WORKERS     = 2
OUTER_FOLDS     = 5   # Maximize folds for robustness
OPTUNA_TRIALS   = 10  # Maximize trials since dataset is small
FL_ROUNDS       = 20  # reduced: 20->10 (convergence happens in first 10 rounds)         # more FL rounds -> better convergence
FL_LOCAL_EPOCHS = 5  # increased from 3 for better convergence
FEDPROX_MU      = 0.01

# Training recipe for ~96% accuracy
MAX_EPOCHS      = 25         # more epochs (early stopping prevents overfit)
WARMUP_EPOCHS   = 5          # LR warmup to stabilise early training
BASE_LR         = 2e-4       # higher initial LR with warmup
MIN_LR          = 1e-6       # cosine annealing floor
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1        # prevents overconfidence
EMA_DECAY       = 0.9998     # Exponential Moving Average for smooth curves
GRAD_CLIP       = 1.0        # gradient clipping
MIXUP_ALPHA     = 0.2        # Mixup augmentation
PATIENCE        = 8         # more patience with cosine schedule

SAVE_DIR = Path("./outputs")
for sub in ["models", "figures", "results", "xai"]:
    (SAVE_DIR / sub).mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)
print(f"Device : {device}")
print(f"Classes: {CLASSES}")

Device : cuda
Classes: ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']


## CHECKPOINT RESUME LOADER

> **Run this cell to skip heavy training and resume from saved checkpoints.**
>
> If your notebook crashed or you want to jump straight to evaluation/XAI:
> 1. Run **Sections 1-3** (imports, config, preprocessing) as normal
> 2. Run **this cell** -- it loads all trained models and DataFrames
> 3. Jump to **any section** (XAI, fairness, error analysis, etc.)

| Checkpoint | What it restores | Skip to |
|---|---|---|
| `ckpt_data.pkl` | DataFrames (train/val/test) | Section 8 onwards |
| `ckpt_best_params.json` | Optuna best hyperparams | Section 9 onwards |
| `ckpt_cent_model.pt` | Centralized model weights | Section 11 onwards |
| `ckpt_fl_models/*.pt` | All FL model weights | Section 11 onwards |
| `ckpt_all_evals.json` | All metric results | Section 12 onwards |
| `ckpt_fold_histories.pkl` | CV training histories | Section 14 onwards |


In [3]:
# ====================================================================
# UNIVERSAL CHECKPOINT RESUME LOADER
# Run this cell INSTEAD of Sections 2, 8, 9, 10, 11
# to restore all variables from saved checkpoints.
# ====================================================================

import pickle, json as _json, os
from pathlib import Path

CKPT_DIR = Path('./outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def ckpt_path(name):
    return CKPT_DIR / name

loaded = []
skipped = []

# ── 1. DataFrames (replaces Section 2) ───────────────────────────────────
if ckpt_path('ckpt_data.pkl').exists():
    with open(ckpt_path('ckpt_data.pkl'), 'rb') as f:
        _data = pickle.load(f)
    df_oasis            = _data['df_oasis']
    df_external         = _data['df_external']
    df_train            = _data['df_train']
    df_val              = _data['df_val']
    df_test_internal    = _data['df_test_internal']
    df_test_external    = _data['df_test_external']
    loaded.append('DataFrames (df_train/val/test_internal/test_external)')
    print(f'[LOADED] DataFrames: train={len(df_train)} val={len(df_val)} '
          f'test_int={len(df_test_internal)} test_ext={len(df_test_external)}')
else:
    skipped.append('DataFrames -- run Section 2 first')

# ── 2. Best Optuna hyperparams (replaces Section 8) ──────────────────────
if ckpt_path('ckpt_best_params.json').exists():
    with open(ckpt_path('ckpt_best_params.json')) as f:
        best_params = _json.load(f)
    loaded.append('best_params (Optuna)')
    print(f'[LOADED] best_params: {best_params}')
else:
    skipped.append('best_params -- run Section 8 first')

# ── 3. Centralized model (replaces Section 9) ────────────────────────────
if ckpt_path('ckpt_cent_model.pt').exists():
    try:
        cent_model = DualChannelModel(NUM_CLASSES).to(device)
        cent_model.load_state_dict(
            torch.load(ckpt_path('ckpt_cent_model.pt'), map_location=device))
        cent_model.eval()
        loaded.append('cent_model')
        print('[LOADED] cent_model from ckpt_cent_model.pt')
    except Exception as e:
        print(f'[WARN] cent_model load failed: {e}')
        skipped.append('cent_model -- error loading')
else:
    skipped.append('cent_model -- run Section 9 first')

# ── 4. Fold models & histories (replaces Section 9) ──────────────────────
if ckpt_path('ckpt_fold_histories.pkl').exists():
    with open(ckpt_path('ckpt_fold_histories.pkl'), 'rb') as f:
        _fold_data = pickle.load(f)
    fold_histories = _fold_data.get('fold_histories', [])
    fold_mdls      = _fold_data.get('fold_mdls', [])
    loaded.append('fold_histories + fold_mdls')
    print(f'[LOADED] fold_histories: {len(fold_histories)} folds')
else:
    skipped.append('fold_histories -- run Section 9 first')

# ── 5. FL models (replaces Section 10) ───────────────────────────────────
fl_models = {}
for algo in ['FedAvg', 'FedProx', 'FedBN']:
    pt_path = ckpt_path(f'ckpt_fl_{algo.lower()}.pt')
    if pt_path.exists():
        try:
            mdl = DualChannelModel(NUM_CLASSES).to(device)
            mdl.load_state_dict(torch.load(pt_path, map_location=device))
            mdl.eval()
            fl_models[algo] = mdl
            loaded.append(f'fl_models[{algo}]')
            print(f'[LOADED] fl_models["{algo}"] from {pt_path.name}')
        except Exception as e:
            print(f'[WARN]  fl_models["{algo}"] load failed: {e}')
    else:
        skipped.append(f'fl_models[{algo}] -- run Section 10 first')

# ── 6. Evaluation results (replaces Section 11) ───────────────────────────
if ckpt_path('ckpt_all_evals.json').exists():
    with open(ckpt_path('ckpt_all_evals.json')) as f:
        all_evals = _json.load(f)
    loaded.append('all_evals')
    print(f'[LOADED] all_evals: {list(all_evals.keys())}')
else:
    skipped.append('all_evals -- run Section 11 first')

# ── Summary ────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('  CHECKPOINT LOADER SUMMARY')
print('=' * 60)
print(f'  Loaded  ({len(loaded)}): ')
for item in loaded:
    print(f'    [OK]  {item}')
if skipped:
    print(f'  Missing ({len(skipped)}): ')
    for item in skipped:
        print(f'    [--]  {item}')
print()
if len(skipped) == 0:
    print('  ALL CHECKPOINTS LOADED. You can jump to any section.')
else:
    first_missing = skipped[0].split(' --')[0]
    print(f'  Partial restore. Run missing sections, then re-run this cell.')
print('=' * 60)



  CHECKPOINT LOADER SUMMARY
  Loaded  (0): 
  Missing (8): 
    [--]  DataFrames -- run Section 2 first
    [--]  best_params -- run Section 8 first
    [--]  cent_model -- run Section 9 first
    [--]  fold_histories -- run Section 9 first
    [--]  fl_models[FedAvg] -- run Section 10 first
    [--]  fl_models[FedProx] -- run Section 10 first
    [--]  fl_models[FedBN] -- run Section 10 first
    [--]  all_evals -- run Section 11 first

  Partial restore. Run missing sections, then re-run this cell.


## Section 2 — Data Loading & Patient-Level Split

In [4]:
def extract_patient_id(filename: str) -> str:
    
    m = re.match(r"(OAS\d+_\d+_MR\d+)", filename)
    return m.group(1) if m else os.path.splitext(filename)[0]

def build_df_oasis(root: str) -> pd.DataFrame:
    """
    Load OASIS dataset.
    OASIS_FOLDER_MAP keys = actual disk folder names on Kaggle.
    Falls back to class name if disk name not found.
    """
    rows = []
    if not os.path.isdir(root):
        print(f'[WARN] OASIS root not found: {root}')
        return pd.DataFrame(columns=['path','label','class_name','patient_id','source'])
    for disk_name, class_name in OASIS_FOLDER_MAP.items():
        folder = os.path.join(root, disk_name)
        if not os.path.isdir(folder):
            folder = os.path.join(root, class_name)   # fallback: try class name
        if not os.path.isdir(folder):
            print(f'[WARN] OASIS: folder not found: {disk_name} or {class_name}')
            continue
        lbl = CLASS2IDX[class_name]
        for fn in os.listdir(folder):
            if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                rows.append({'path': os.path.join(folder, fn),
                             'label': lbl, 'class_name': class_name,
                             'patient_id': extract_patient_id(fn),
                             'source': 'oasis'})
    df = pd.DataFrame(rows)
    print(f'[OASIS] {len(df)} images | {df["patient_id"].nunique()} patients')
    print(df['class_name'].value_counts())
    return df

def build_df_external(root: str) -> pd.DataFrame:
    """
    Load external dataset.
    Uses case-insensitive folder matching to handle 'Nondemented' vs 'NonDemented'.
    """
    rows = []
    # Build case-insensitive lookup: lowercase_folder_name -> actual_path
    if not os.path.isdir(root):
        print(f'[WARN] External root not found: {root}')
        return pd.DataFrame(columns=['path','label','class_name','patient_id','source'])
    folder_map = {f.lower(): f for f in os.listdir(root)
                  if os.path.isdir(os.path.join(root, f))}
    for class_name in CLASSES:
        # Try exact match first, then case-insensitive
        actual_folder = folder_map.get(class_name.lower())
        if actual_folder is None:
            print(f'[WARN] External: no folder for {class_name}'); continue
        folder = os.path.join(root, actual_folder)
        lbl = CLASS2IDX[class_name]
        for fn in os.listdir(folder):
            if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                rows.append({'path': os.path.join(folder, fn),
                             'label': lbl, 'class_name': class_name,
                             'patient_id': f'ext_{fn}',
                             'source': 'external'})
    df = pd.DataFrame(rows)
    print(f'[External] {len(df)} images')
    if len(df) > 0:
        print(df['class_name'].value_counts())
    return df

def patient_level_split(df, test_size=0.15, val_size=0.15, seed=SEED):
    """Split by patient_id — zero leakage guaranteed."""
    pid_label = (df.groupby("patient_id")["label"]
                   .agg(lambda x: x.mode()[0]).reset_index()
                   .rename(columns={"label": "plabel"}))
    pids, plabels = pid_label["patient_id"].values, pid_label["plabel"].values

    pids_tr, pids_te = train_test_split(
        pids, test_size=test_size, stratify=plabels, random_state=seed)
    plabels_tr = pid_label.set_index("patient_id").loc[pids_tr, "plabel"].values
    pids_tr, pids_va = train_test_split(
        pids_tr, test_size=val_size/(1-test_size),
        stratify=plabels_tr, random_state=seed)

    df_tr = df[df["patient_id"].isin(pids_tr)].reset_index(drop=True)
    df_va = df[df["patient_id"].isin(pids_va)].reset_index(drop=True)
    df_te = df[df["patient_id"].isin(pids_te)].reset_index(drop=True)
    print(f"Train {len(df_tr)} | Val {len(df_va)} | Test {len(df_te)}")
    return df_tr, df_va, df_te

df_oasis    = build_df_oasis(OASIS_ROOT)
df_external = build_df_external(EXTERNAL_ROOT)
df_train, df_val, df_test_internal = patient_level_split(df_external)
df_test_external = df_oasis   


print("\n── OASIS dataset ──")
print(df_oasis["label"].value_counts().rename(index=dict(enumerate(CLASSES))))
print(f"Total OASIS images : {len(df_oasis)}")
display(df_oasis.head(3))

print("\n── External dataset ──")
print(df_external["label"].value_counts().rename(index=dict(enumerate(CLASSES))))
print(f"Total External images : {len(df_external)}")
display(df_external.head(3))

print(f"\nSplit → Train:{len(df_train)} | Val:{len(df_val)} "
      f"| Test-Internal:{len(df_test_internal)} | Test-External:{len(df_test_external)}")

[OASIS] 86437 images | 366 patients
class_name
NonDemented         67222
VeryMildDemented    13725
MildDemented         5002
ModerateDemented      488
Name: count, dtype: int64
[External] 6400 images
class_name
NonDemented         3200
VeryMildDemented    2240
MildDemented         896
ModerateDemented      64
Name: count, dtype: int64
Train 4489 | Val 960 | Test 951

── OASIS dataset ──
label
NonDemented         67222
VeryMildDemented    13725
MildDemented         5002
ModerateDemented      488
Name: count, dtype: int64
Total OASIS images : 86437


,path,label,class_name,patient_id,source
0,/kaggle/input/datasets/ninadaithal/imagesoasis...,0,NonDemented,OAS1_0302_MR1,oasis
1,/kaggle/input/datasets/ninadaithal/imagesoasis...,0,NonDemented,OAS1_0114_MR1,oasis
2,/kaggle/input/datasets/ninadaithal/imagesoasis...,0,NonDemented,OAS1_0150_MR1,oasis



── External dataset ──
label
NonDemented         3200
VeryMildDemented    2240
MildDemented         896
ModerateDemented      64
Name: count, dtype: int64
Total External images : 6400


,path,label,class_name,patient_id,source
0,/kaggle/input/datasets/uraninjo/augmented-alzh...,0,NonDemented,ext_nonDem706.jpg,external
1,/kaggle/input/datasets/uraninjo/augmented-alzh...,0,NonDemented,ext_nonDem366.jpg,external
2,/kaggle/input/datasets/uraninjo/augmented-alzh...,0,NonDemented,ext_nonDem222.jpg,external



Split → Train:4489 | Val:960 | Test-Internal:951 | Test-External:86437


### Checkpoint Save -- After Section 2 (Data)
Saves all DataFrames. **Run this once after Section 2 completes.**
After saving, you can reload them via the Resume Loader to skip Section 2 next time.


In [5]:
# CHECKPOINT: Save DataFrames after Section 2
import pickle
from pathlib import Path
CKPT_DIR = Path('./outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

_data_ckpt = {
    'df_oasis':         df_oasis,
    'df_external':      df_external,
    'df_train':         df_train,
    'df_val':           df_val,
    'df_test_internal': df_test_internal,
    'df_test_external': df_test_external,
}
with open(CKPT_DIR / 'ckpt_data.pkl', 'wb') as _f:
    pickle.dump(_data_ckpt, _f, protocol=pickle.HIGHEST_PROTOCOL)
print('[CHECKPOINT SAVED] ckpt_data.pkl')
print(f'  train={len(df_train)} | val={len(df_val)} | '
      f'test_internal={len(df_test_internal)} | test_external={len(df_test_external)}')


[CHECKPOINT SAVED] ckpt_data.pkl
  train=4489 | val=960 | test_internal=951 | test_external=86437


## Section 3 — Preprocessing & Augmentation

## Section 3a -- Deep MRI Preprocessing (Proposal Step 5)

Standard resize+normalise is **insufficient** for a CVPR-level pipeline.  
This section adds:
- **Skull-strip simulation** (elliptical mask removes non-brain pixels)
- **Bias-field correction** (N4-style Gaussian low-freq field removal)
- **CLAHE** adaptive contrast enhancement
- **Gaussian denoising** (acquisition noise reduction)
- **ROI/Hippocampus-guided cropping** (medial temporal lobe focus)

All steps are applied to **both train and eval** transforms;
augmentation (flips, jitter, erasing) is **training-only**.


In [6]:
# ====================================================================
# STEP 5 -- NOVEL AND ROBUST MRI PREPROCESSING (SAFE VERSION)
# Fixed: sigma, skull-strip safety, correct order, no image destruction
# ====================================================================

import cv2
import numpy as np
from PIL import Image


class SkullStripSimulation:
    """
    SAFE skull-strip simulation for 2D MRI slices.
    Uses a SOFT elliptical weight (not hard zero) so brain tissue at
    the edges is attenuated rather than destroyed.
    Kaggle/OASIS 2D slices are already preprocessed, so this is
    applied with blend_strength=0.1 to avoid damaging existing features.
    Set enabled=False to disable entirely if images are pre-stripped.
    """
    def __init__(self, blend_strength=0.1, enabled=True):  # ↓ was 0.3 — lighter strip preserves more features
        self.blend   = blend_strength
        self.enabled = enabled

    def __call__(self, img):
        if not self.enabled:
            return img
        try:
            arr = np.array(img.convert('RGB')).astype(np.float32)
            h, w = arr.shape[:2]
            # Soft Gaussian weight mask centred on brain
            Y, X = np.ogrid[:h, :w]
            cx, cy = w / 2, h / 2
            rx, ry = w * 0.46, h * 0.48
            ellipse_dist = ((X - cx) / rx) ** 2 + ((Y - cy) / ry) ** 2
            # Smooth sigmoid falloff at the ellipse boundary
            weight = 1.0 / (1.0 + np.exp(8.0 * (ellipse_dist - 1.0)))
            weight = weight[:, :, np.newaxis]  # (H,W,1)
            # Blend: inside->original, outside->attenuated (not zeroed)
            blended = arr * (weight + (1 - weight) * (1 - self.blend))
            return Image.fromarray(blended.clip(0, 255).astype(np.uint8))
        except Exception:
            return img  # fallback: return unchanged


class BiasFieldCorrection:
    """
    Approximate bias-field correction using Gaussian smoothing.
    FIXED: sigma=2.0 -> kernel=13x13 (safe for 224x224 images).
    sigma=40 was wrong -- it produced a 241x241 kernel which averaged
    the whole image to one colour, destroying all spatial information.
    sigma=2 correctly estimates local intensity inhomogeneity.
    """
    def __init__(self, sigma=2.0):
        self.sigma = sigma

    def __call__(self, img):
        try:
            arr = np.array(img.convert('RGB')).astype(np.float32)
            ksize = max(3, int(6 * self.sigma) | 1)  # odd, at least 3
            bias = np.zeros_like(arr)
            for c in range(3):
                bias[..., c] = cv2.GaussianBlur(arr[..., c], (ksize, ksize), self.sigma)
            # Divide by local field estimate (keeps global scale)
            corrected = arr / (bias + 1e-3) * bias.mean(axis=(0, 1), keepdims=True).clip(1e-3)
            return Image.fromarray(corrected.clip(0, 255).astype(np.uint8))
        except Exception:
            return img


class CLAHEEnhancement:
    """
    CLAHE applied to L channel in LAB colour space.
    clip_limit=2.5 (conservative) to avoid over-amplifying noise.
    """
    def __init__(self, clip_limit=2.5, tile_grid=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)

    def __call__(self, img):
        try:
            arr = np.array(img.convert('RGB'))
            lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
            lab[:, :, 0] = self.clahe.apply(lab[:, :, 0])
            return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))
        except Exception:
            return img


class GaussianDenoising:
    """Mild Gaussian denoising (sigma=0.5). Conservative and safe."""
    def __init__(self, sigma=0.5):
        self.sigma = sigma

    def __call__(self, img):
        try:
            arr = np.array(img.convert('RGB'))
            return Image.fromarray(cv2.GaussianBlur(arr, (3, 3), self.sigma))
        except Exception:
            return img


class ROIGuidedCrop:
    """
    Gentle hippocampus/medial-temporal-lobe-guided crop.
    FIXED: roi_frac=0.48 (keeps 96% of image on each side).
    Previous value 0.42 cropped too aggressively and, combined with
    skull-strip zeroing + CropBlackBorders, shrank images to tiny patches.
    This version is a subtle refocus, not a destructive crop.
    """
    def __init__(self, cx_frac=0.50, cy_frac=0.53, roi_frac=0.48):
        self.cx = cx_frac; self.cy = cy_frac; self.roi = roi_frac

    def __call__(self, img):
        try:
            w, h = img.size
            hw = int(w * self.roi); hh = int(h * self.roi)
            cx = int(w * self.cx);  cy = int(h * self.cy)
            x0 = max(0, cx - hw); y0 = max(0, cy - hh)
            x1 = min(w, cx + hw); y1 = min(h, cy + hh)
            return img.crop((x0, y0, x1, y1))
        except Exception:
            return img


class CropBlackBorders:
    """Remove black MRI borders."""
    def __init__(self, thr=10, pad=5): self.thr = thr; self.pad = pad
    def __call__(self, img):
        try:
            arr  = np.array(img)
            mask = (arr.sum(axis=2) > self.thr) if arr.ndim == 3 else (arr > self.thr)
            if not mask.any(): return img
            ys, xs = np.where(mask)
            y0 = max(0, ys.min() - self.pad); y1 = min(arr.shape[0]-1, ys.max() + self.pad)
            x0 = max(0, xs.min() - self.pad); x1 = min(arr.shape[1]-1, xs.max() + self.pad)
            return img.crop((x0, y0, x1+1, y1+1))
        except Exception:
            return img


class EnsureRGB:
    def __call__(self, img): return img.convert('RGB')


class ZScorePerImage:
    """Per-image Z-score normalisation -- scanner-invariant."""
    def __call__(self, x):
        m = x.mean(dim=(1, 2), keepdim=True)
        s = x.std(dim=(1, 2), keepdim=True).clamp(min=1e-6)
        return (x - m) / s


# ── CORRECT pipeline order ────────────────────────────────────────────────
# 1. EnsureRGB: convert any grayscale to RGB
# 2. CropBlackBorders: remove scanner background BEFORE skull strip
# 3. SkullStripSimulation: SOFT blend (not hard zero), blend_strength=0.1
# 4. BiasFieldCorrection: sigma=2.0 (FIXED from 40.0)
# 5. CLAHEEnhancement: clip_limit=2.5 (conservative)
# 6. GaussianDenoising: sigma=0.5
# 7. ROIGuidedCrop: roi_frac=0.48 (gentle, not destructive)
# 8. Resize to 256x256
# 9. Augmentation (training only)
# 10. ToTensor + ZScorePerImage

train_transform = transforms.Compose([
    EnsureRGB(),
    CropBlackBorders(),                          # remove background first
    SkullStripSimulation(blend_strength=0.1),    # soft blend, not hard zero
    BiasFieldCorrection(sigma=2.0),              # FIXED: sigma=2 not 40
    CLAHEEnhancement(clip_limit=2.5),  # ↓ was 3.0 — 2.5 enhances without over-amplifying noise            # conservative
    GaussianDenoising(sigma=0.5),
    ROIGuidedCrop(cx_frac=0.50, cy_frac=0.53, roi_frac=0.48),  # gentle
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.5, 1.5))], p=0.3),
    transforms.RandAugment(num_ops=2, magnitude=7),  # auto-selects best augmentations
    transforms.ToTensor(),
    ZScorePerImage(),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.10)),  # must be after ToTensor()
])

eval_transform = transforms.Compose([
    EnsureRGB(),
    CropBlackBorders(),
    SkullStripSimulation(blend_strength=0.1),
    BiasFieldCorrection(sigma=2.0),
    CLAHEEnhancement(clip_limit=2.5),  # ↓ was 3.0 — 2.5 enhances without over-amplifying noise
    GaussianDenoising(sigma=0.5),
    ROIGuidedCrop(cx_frac=0.50, cy_frac=0.53, roi_frac=0.48),
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    ZScorePerImage(),
])


def visualise_preprocessing(df, n_samples=3, seed=42):
    """Show side-by-side preprocessing stages for n sample images."""
    samples = df.sample(min(n_samples, len(df)), random_state=seed)
    stages = [
        ('1.Original',       EnsureRGB()),
        ('2.CropBB',         transforms.Compose([EnsureRGB(), CropBlackBorders()])),
        ('3.SkullStrip(soft)',transforms.Compose([EnsureRGB(), CropBlackBorders(),
                                                  SkullStripSimulation(blend_strength=0.1)])),
        ('4.BiasCorrect',    transforms.Compose([EnsureRGB(), CropBlackBorders(),
                                                  SkullStripSimulation(blend_strength=0.1),
                                                  BiasFieldCorrection(sigma=2.0)])),
        ('5.CLAHE+Denoise',  transforms.Compose([EnsureRGB(), CropBlackBorders(),
                                                  SkullStripSimulation(blend_strength=0.1),
                                                  BiasFieldCorrection(sigma=2.0),
                                                  CLAHEEnhancement(clip_limit=2.5),  # ↓ was 3.0 — 2.5 enhances without over-amplifying noise
                                                  GaussianDenoising(sigma=0.5),
                                                  ROIGuidedCrop(roi_frac=0.48),
                                                  transforms.Resize((IMG_SIZE, IMG_SIZE))])),
    ]
    fig, axes = plt.subplots(len(samples), len(stages),
                             figsize=(4 * len(stages), 3.5 * len(samples)))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]
    for r, (_, row) in enumerate(samples.iterrows()):
        img_pil = Image.open(row['path'])
        for c, (stage_name, tfm) in enumerate(stages):
            processed = np.array(tfm(img_pil))
            axes[r, c].imshow(processed, cmap='gray' if processed.ndim == 2 else None)
            if r == 0:
                axes[r, c].set_title(stage_name, fontsize=8, fontweight='bold')
            if c == 0:
                axes[r, c].set_ylabel(CLASSES[row['label']], fontsize=7,
                                      rotation=0, labelpad=60, va='center')
            axes[r, c].axis('off')
    plt.suptitle('MRI Preprocessing -- Stage Visualisation (FIXED)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'figures' / 'preprocessing_stages.png',
                dpi=150, bbox_inches='tight')
    plt.close()
    print('[Step 5] Preprocessing visualisation saved.')


try:
    visualise_preprocessing(df_train, n_samples=3)
except NameError:
    print('[Step 5] df_train not yet defined -- run after Section 2.')

print('[Step 5] FIXED preprocessing pipeline configured.')
print('  Key fixes: sigma=2.0 (was 40), soft skull strip (was hard zero),')
print('  roi_frac=0.48 (was 0.42), CropBB BEFORE skull strip.')
print('  Train steps:', [type(t).__name__ for t in train_transform.transforms])


[Step 5] Preprocessing visualisation saved.
[Step 5] FIXED preprocessing pipeline configured.
  Key fixes: sigma=2.0 (was 40), soft skull strip (was hard zero),
  roi_frac=0.48 (was 0.42), CropBB BEFORE skull strip.
  Train steps: ['EnsureRGB', 'CropBlackBorders', 'SkullStripSimulation', 'BiasFieldCorrection', 'CLAHEEnhancement', 'GaussianDenoising', 'ROIGuidedCrop', 'Resize', 'RandomResizedCrop', 'RandomHorizontalFlip', 'RandomRotation', 'ColorJitter', 'RandomApply', 'RandAugment', 'ToTensor', 'ZScorePerImage', 'RandomErasing']


## Section 3b -- Multi-Dataset Harmonization & Domain-Shift Evaluation
### Proposal Steps 4 & 27

Implements:
- **ComBat-style** statistical harmonisation (site-mean/std correction)
- **KS test** to quantify pixel-intensity distribution shift between sites
- **MMD** (Maximum Mean Discrepancy) with RBF kernel
- Before/after distribution visualisation
- FedBN keeps site-specific BN statistics local in federated training


In [7]:
# ====================================================================
# STEPS 4 & 27 -- MULTI-DATASET HARMONIZATION & DOMAIN-SHIFT EVALUATION
# ====================================================================

from scipy import stats


def combat_harmonise(df_source, df_target, ref_source='oasis'):
    """
    ComBat-style harmonisation.
    Shifts pixel intensity statistics of each non-reference site toward
    the reference site mean/std.  Correction factors are stored as
    'site_mean_offset' and 'site_std_scale' columns in the returned df.
    These columns can be used in a custom Dataset to rescale tensors.
    """
    SAMPLE_N = min(200, len(df_source))
    site_stats = {}
    for site in df_source['source'].unique():
        df_s = df_source[df_source['source'] == site].sample(
            min(SAMPLE_N, (df_source['source'] == site).sum()), random_state=42)
        means, stds = [], []
        for _, row in df_s.iterrows():
            try:
                arr = np.array(Image.open(row['path']).convert('L'), dtype=np.float32)
                means.append(arr.mean()); stds.append(arr.std() + 1e-6)
            except Exception:
                pass
        if means:
            site_stats[site] = {'mean': float(np.mean(means)), 'std': float(np.mean(stds))}

    if ref_source not in site_stats:
        ref_source = list(site_stats.keys())[0]
    ref_mean = site_stats[ref_source]['mean']
    ref_std  = site_stats[ref_source]['std']

    print(f'[Harmonisation] Reference site: "{ref_source}"  '
          f'mean={ref_mean:.2f}  std={ref_std:.2f}')
    for site, st in site_stats.items():
        offset = ref_mean - st['mean']; scale = ref_std / st['std']
        print(f'  {site:<14}: mean={st["mean"]:.2f}  std={st["std"]:.2f}  '
              f'offset={offset:+.2f}  scale={scale:.3f}')

    df_out = df_target.copy()
    for site, st in site_stats.items():
        mask = df_out['source'] == site
        df_out.loc[mask, 'site_mean_offset'] = ref_mean - st['mean']
        df_out.loc[mask, 'site_std_scale']   = ref_std  / st['std']
    df_out['site_mean_offset'] = df_out.get('site_mean_offset', pd.Series([0.0]*len(df_out))).fillna(0.0)
    df_out['site_std_scale']   = df_out.get('site_std_scale',   pd.Series([1.0]*len(df_out))).fillna(1.0)
    return df_out


def mmd_rbf(X, Y, gamma=1.0):
    """Unbiased MMD squared with RBF kernel. Lower = better alignment."""
    from sklearn.metrics.pairwise import rbf_kernel
    XX = rbf_kernel(X, X, gamma); YY = rbf_kernel(Y, Y, gamma)
    XY = rbf_kernel(X, Y, gamma)
    n, m = len(X), len(Y)
    return float(
        (XX.sum() - np.diag(XX).sum()) / (n * (n - 1)) +
        (YY.sum() - np.diag(YY).sum()) / (m * (m - 1)) -
        2 * XY.mean()
    )


def compute_domain_stats(df, n=300):
    """Sample n images and return per-image mean + std arrays."""
    sample = df.sample(min(n, len(df)), random_state=42)
    means, stds = [], []
    for _, row in sample.iterrows():
        try:
            arr = np.array(Image.open(row['path']).convert('L'), dtype=np.float32)
            means.append(arr.mean()); stds.append(arr.std())
        except Exception:
            pass
    return np.array(means), np.array(stds)


try:
    print('[Domain Shift] Computing pixel statistics...')
    means_oasis, stds_oasis = compute_domain_stats(df_oasis)
    means_ext,   stds_ext   = compute_domain_stats(df_external)

    print(f'  OASIS    -- mean: {means_oasis.mean():.2f} +/- {means_oasis.std():.2f}  '
          f'std: {stds_oasis.mean():.2f} +/- {stds_oasis.std():.2f}')
    print(f'  External -- mean: {means_ext.mean():.2f}   +/- {means_ext.std():.2f}  '
          f'std: {stds_ext.mean():.2f}   +/- {stds_ext.std():.2f}')

    ks_stat_m, ks_p_m = stats.ks_2samp(means_oasis, means_ext)
    ks_stat_s, ks_p_s = stats.ks_2samp(stds_oasis,  stds_ext)
    print(f'  KS test (pixel mean): stat={ks_stat_m:.4f}  p={ks_p_m:.4f}  '
          f'{"** SIGNIFICANT **" if ks_p_m < 0.05 else "No significant shift"}')
    print(f'  KS test (pixel std) : stat={ks_stat_s:.4f}  p={ks_p_s:.4f}  '
          f'{"** SIGNIFICANT **" if ks_p_s < 0.05 else "No significant shift"}')

    X_oasis = np.stack([means_oasis, stds_oasis], axis=1)
    X_ext   = np.stack([means_ext,   stds_ext],   axis=1)
    mmd_val = mmd_rbf(X_oasis, X_ext, gamma=1.0 / (X_oasis.var() + 1e-6))
    print(f'  MMD2 (RBF)          : {mmd_val:.6f}  '
          f'{"high shift" if mmd_val > 0.01 else "low shift"}')

    # Visualise domain shift
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(means_oasis, bins=40, alpha=0.7, label='OASIS',    color='#3498db', density=True)
    axes[0].hist(means_ext,   bins=40, alpha=0.7, label='External', color='#e74c3c', density=True)
    axes[0].set(title=f'Pixel Mean (KS p={ks_p_m:.4f})',
                xlabel='Mean Intensity', ylabel='Density')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].hist(stds_oasis, bins=40, alpha=0.7, label='OASIS',    color='#3498db', density=True)
    axes[1].hist(stds_ext,   bins=40, alpha=0.7, label='External', color='#e74c3c', density=True)
    axes[1].set(title=f'Pixel Std (KS p={ks_p_s:.4f})',
                xlabel='Std Intensity', ylabel='Density')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle('Domain Shift Evaluation: OASIS vs External Dataset', fontsize=12)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'figures' / 'domain_shift.png', dpi=150)
    plt.close()
    print('[Step 4/27] Domain-shift plot saved.')

    # Apply ComBat harmonisation
    print('[Harmonisation] Applying ComBat-style correction...')
    df_train         = combat_harmonise(df_external, df_train, ref_source='external')
    df_val           = combat_harmonise(df_external, df_val, ref_source='external')
    df_test_internal = combat_harmonise(df_external, df_test_internal, ref_source='external')
    df_test_external = combat_harmonise(df_external, df_test_external, ref_source='external')

    harm_summary = {
        'mmd_before': mmd_val, 'ks_mean_stat': float(ks_stat_m),
        'ks_mean_p': float(ks_p_m), 'ks_std_stat': float(ks_stat_s),
        'ks_std_p': float(ks_p_s),
        'oasis_pixel_mean': float(means_oasis.mean()),
        'external_pixel_mean': float(means_ext.mean()),
    }
    import json as _json
    with open(SAVE_DIR / 'results' / 'harmonisation_summary.json', 'w') as _f:
        _json.dump(harm_summary, _f, indent=2)
    print('[Step 4/27] Harmonisation complete.\n')

except NameError as e:
    print(f'[Step 4/27] DataFrames not yet defined: {e}. Run after Section 2.')


[Domain Shift] Computing pixel statistics...
  OASIS    -- mean: 41.14 +/- 6.60  std: 44.47 +/- 3.96
  External -- mean: 72.17   +/- 6.97  std: 83.41   +/- 6.94
  KS test (pixel mean): stat=0.9667  p=0.0000  ** SIGNIFICANT **
  KS test (pixel std) : stat=0.9967  p=0.0000  ** SIGNIFICANT **
  MMD2 (RBF)          : 0.534371  high shift
[Step 4/27] Domain-shift plot saved.
[Harmonisation] Applying ComBat-style correction...
[Harmonisation] Reference site: "external"  mean=71.86  std=83.11
  external      : mean=71.86  std=83.11  offset=+0.00  scale=1.000
[Harmonisation] Reference site: "external"  mean=71.86  std=83.11
  external      : mean=71.86  std=83.11  offset=+0.00  scale=1.000
[Harmonisation] Reference site: "external"  mean=71.86  std=83.11
  external      : mean=71.86  std=83.11  offset=+0.00  scale=1.000
[Harmonisation] Reference site: "external"  mean=71.86  std=83.11
  external      : mean=71.86  std=83.11  offset=+0.00  scale=1.000
[Step 4/27] Harmonisation complete.



In [8]:
# Helper classes kept for backward compatibility (used in cell 10)
# train_transform and eval_transform are defined in Section 3a (cell 10)
# with full deep preprocessing pipeline. Do NOT redefine them here.

class EnsureRGB:
    def __call__(self, img): return img.convert('RGB')

class CropBlackBorders:
    """Remove black MRI borders by finding the bounding box of non-zero pixels."""
    def __init__(self, thr=10, pad=5): self.thr = thr; self.pad = pad
    def __call__(self, img):
        arr  = np.array(img)
        mask = (arr.sum(axis=2) > self.thr) if arr.ndim == 3 else (arr > self.thr)
        if not mask.any(): return img
        ys, xs = np.where(mask)
        y0 = max(0, ys.min() - self.pad); y1 = min(arr.shape[0]-1, ys.max() + self.pad)
        x0 = max(0, xs.min() - self.pad); x1 = min(arr.shape[1]-1, xs.max() + self.pad)
        return img.crop((x0, y0, x1+1, y1+1))

class ZScorePerImage:
    """Per-image Z-score normalisation -- scanner-invariant."""
    def __call__(self, x):
        m = x.mean(dim=(1, 2), keepdim=True)
        s = x.std(dim=(1, 2), keepdim=True).clamp(min=1e-6)
        return (x - m) / s

# Confirm transforms from cell 10 are active
print('Helper classes ready. train_transform / eval_transform from Section 3a are in effect.')
print('  Train:', [type(t).__name__ for t in train_transform.transforms])
print('  Eval :', [type(t).__name__ for t in eval_transform.transforms])


Helper classes ready. train_transform / eval_transform from Section 3a are in effect.
  Train: ['EnsureRGB', 'CropBlackBorders', 'SkullStripSimulation', 'BiasFieldCorrection', 'CLAHEEnhancement', 'GaussianDenoising', 'ROIGuidedCrop', 'Resize', 'RandomResizedCrop', 'RandomHorizontalFlip', 'RandomRotation', 'ColorJitter', 'RandomApply', 'RandAugment', 'ToTensor', 'ZScorePerImage', 'RandomErasing']
  Eval : ['EnsureRGB', 'CropBlackBorders', 'SkullStripSimulation', 'BiasFieldCorrection', 'CLAHEEnhancement', 'GaussianDenoising', 'ROIGuidedCrop', 'Resize', 'CenterCrop', 'ToTensor', 'ZScorePerImage']


## Section 4 — Dataset Class & DataLoaders

In [9]:
class AlzheimerDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df=df.reset_index(drop=True); self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        with Image.open(row["path"]) as img:
            img = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return img, int(row["label"]), row["path"]

def make_loader(df_part, transform, shuffle, bs=BATCH_SIZE):
    ds = AlzheimerDataset(df_part, transform)
    if shuffle:  # WeightedRandomSampler — fixes 137:1 class imbalance
        labels  = df_part["label"].values
        counts  = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
        counts  = np.maximum(counts, 1)        # avoid div by zero
        weights = 1.0 / counts[labels]         # inverse-frequency weights
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=torch.DoubleTensor(weights),
            num_samples=len(weights), replacement=True)
        return DataLoader(ds, batch_size=bs, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS>0),
                          drop_last=True)   # FIX: never yield a final batch of
                                            # size 1 -- BatchNorm1d in classifier
                                            # crashes in .train() mode on batch=1
    return DataLoader(ds, batch_size=bs, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS>0))

def class_weights(df):
    counts = (df["label"].value_counts().sort_index()
                .reindex(range(NUM_CLASSES), fill_value=0).values)
    counts = np.maximum(counts, 1).astype(np.float32)
    w = counts.sum() / (NUM_CLASSES * counts)
    return torch.tensor(w, device=device)

print("Dataset class ready.")


Dataset class ready.


## Section 5 — Model Architecture
### 5a CBAM

In [10]:
# ================================================================
# SECTION 5 -- MODEL ARCHITECTURE (COMPLETE)
# Dual-Channel ConvNeXt-Small + Swin-V2-S + CBAM
# Fixes: missing backbones, forward_features, Dropout, drop_path
# ================================================================

import timm
import torch.nn as nn
import torch

# ── 5a. CBAM (Channel + Spatial Attention) ──────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        r = max(1, channels // reduction)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(r, channels, bias=False),
        )
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(
            self.fc(self.avg(x)) + self.fc(self.max(x))
        ).unsqueeze(-1).unsqueeze(-1)


class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx  = x.max(dim=1, keepdim=True).values
        return x * self.sig(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_k=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_k)
    def forward(self, x):
        return self.sa(self.ca(x))


# ── 5b. Dual-Channel Model: ConvNeXt-Tiny + Swin-V2-T + CBAM ───────────
class DualChannelAD(nn.Module):
    """
    Dual-channel feature fusion:
      Channel A: ConvNeXt-Small (local texture + CNN inductive bias)
      Channel B: Swin-V2-S     (global context + hierarchical attention)
    Features fused via concatenation, refined by CBAM, classified by MLP head.
    forward_features() exposed for contrastive loss.
    Stochastic depth (drop_path) applied inside timm backbones automatically.
    """
    def __init__(self, num_classes=4, bottleneck_dim=512,
                 dropout=0.4, freeze_ratio=0.5):
        super().__init__()
        # Channel A: ConvNeXt-Tiny
        self.convnext = timm.create_model(
            'convnext_small', pretrained=True, num_classes=0,
            drop_path_rate=0.2)        # ↑ tiny→small: +22M params, deeper net        # stochastic depth
        feat_a = self.convnext.num_features  # 768

        # Channel B: Swin-V2-T
        self.swin = timm.create_model(
            'swinv2_small_window8_256', pretrained=True, num_classes=0,
            drop_path_rate=0.2,   # ↑ tiny→small: +22M params
            img_size=IMG_SIZE)         # stochastic depth
        feat_b = self.swin.num_features      # 768

        feat_total = feat_a + feat_b         # 1536

        # Freeze lower layers of backbones (transfer learning)
        self._freeze_backbones(freeze_ratio)

        # Reshape fusion features for CBAM (treat as 1D 'spatial' of 1x1)
        self.cbam = CBAM(channels=feat_total, reduction=16)
        # Project fused features to bottleneck
        self.projector = nn.Sequential(
            nn.LayerNorm(feat_total),
            nn.Linear(feat_total, bottleneck_dim),
            nn.GELU(),
            nn.Dropout(p=dropout),
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(bottleneck_dim),  # stabilizes final features
            nn.Linear(bottleneck_dim, bottleneck_dim // 2),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(bottleneck_dim // 2, num_classes),
        )

    def _freeze_backbones(self, ratio):
        """Freeze the first `ratio` fraction of layers in each backbone."""
        for backbone in [self.convnext, self.swin]:
            params = list(backbone.parameters())
            n_freeze = int(len(params) * ratio)
            for p in params[:n_freeze]:
                p.requires_grad = False

    def forward_features(self, x):
        """Returns bottleneck embedding (used by contrastive loss)."""
        # Extract features from both channels
        fa = self.convnext(x)           # (B, 768)
        fb = self.swin(x)               # (B, 768)

        # Fuse
        fused = torch.cat([fa, fb], dim=1)  # (B, 1536)

        # CBAM on fused (unsqueeze to (B, C, 1, 1) for spatial ops)
        fused_4d  = fused.unsqueeze(-1).unsqueeze(-1)  # (B, 1536, 1, 1)
        fused_att = self.cbam(fused_4d).squeeze(-1).squeeze(-1)  # (B, 1536)

        # Bottleneck projection
        return self.projector(fused_att)   # (B, bottleneck_dim)

    def forward(self, x):
        feats  = self.forward_features(x)
        return self.classifier(feats)


# Alias used throughout the notebook
DualChannelModel = DualChannelAD

# Quick sanity check
try:
    _m   = DualChannelAD(NUM_CLASSES, 512, 0.4, 0.5).to(device)
    _x   = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    _f   = _m.forward_features(_x)
    _out = _m(_x)
    _tp  = sum(p.numel() for p in _m.parameters()) / 1e6
    _tr  = sum(p.numel() for p in _m.parameters() if p.requires_grad) / 1e6
    print(f'[Model OK] DualChannelAD')
    print(f'  Input    : {list(_x.shape)}')
    print(f'  Features : {list(_f.shape)}')
    print(f'  Output   : {list(_out.shape)}')
    print(f'  Params   : {_tp:.1f}M total | {_tr:.1f}M trainable')
    del _m, _x, _f, _out
    torch.cuda.empty_cache()
except Exception as _e:
    print(f'[Model check failed] {_e}')
    print('  swinv2_small_window8_256 + IMG_SIZE=256: all feature maps divisible by 8')
    print('  IMG_SIZE=256, swinv2_small_window8_256: window8 divides all feature maps perfectly')
print('Model architecture ready.')


model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

[Model OK] DualChannelAD
  Input    : [2, 3, 256, 256]
  Features : [2, 512]
  Output   : [2, 4]
  Params   : 99.6M total | 68.5M trainable
Model architecture ready.


### 5b Dual-Channel ConvNeXt-Tiny + Swin-V2-T Model

In [11]:
# Cell originally contained old DualChannelAD (OLD-REMOVED).
# REMOVED: superseded by the complete DualChannelAD in Section 5 (Cell 21).
# DualChannelAD = the class defined in Cell 21 is already active.
print('DualChannelAD: using upgraded version from Section 5 (Cell 21). OK.')


DualChannelAD: using upgraded version from Section 5 (Cell 21). OK.


## Section 6 — Hybrid Loss Function

In [12]:
class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017).
    Down-weights easy examples, focuses on hard/minority cases.
    gamma=2.0 is standard. Used as one component of HybridLoss.
    """
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, y):
        ce  = F.cross_entropy(logits, y, weight=self.weight, reduction='none')
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


# ── Supervised Contrastive Loss ──────────────────────────────────────
class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., 2020).
    Pulls same-class embeddings together, pushes different-class apart.
    Significantly reduces inter-class confusion especially for adjacent
    AD stages (VeryMild <-> Mild).
    temperature=0.07 is the standard setting from the paper.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        """features: (B, D) L2-normalised embeddings. labels: (B,) int."""
        device = features.device
        features = F.normalize(features, dim=1)
        B = features.shape[0]
        if B < 2:
            return torch.tensor(0.0, device=device)
        sim   = torch.matmul(features, features.T) / self.temperature
        # Mask diagonal (self-similarity)
        mask_eye = torch.eye(B, dtype=torch.bool, device=device)
        sim.masked_fill_(mask_eye, float('-inf'))
        # Positive mask: same class
        labels   = labels.view(-1, 1)
        pos_mask = (labels == labels.T).float()
        pos_mask.fill_diagonal_(0)
        n_pos    = pos_mask.sum(dim=1).clamp(min=1)
        log_prob = F.log_softmax(sim, dim=1)
        loss     = -(pos_mask * log_prob).sum(dim=1) / n_pos
        loss = loss.clamp(min=-50, max=50)  # prevent inf
        loss_val = loss.mean()
        if torch.isnan(loss_val) or torch.isinf(loss_val):
            loss_val = torch.tensor(0.0, requires_grad=True, device=loss.device)
        return loss_val


# ── Upgraded HybridLoss with Soft-Target support for Mixup ───────────
class HybridLoss(nn.Module):
    """
    Hybrid loss = lam1*CE + lam2*Focal + lam3*SupCon
    Upgraded to accept y_soft (soft labels from Mixup).
    When y_soft is provided, CE and Focal use KL-div against soft targets.
    """
    def __init__(self, class_weights=None, lam1=0.4, lam2=0.4, lam3=0.2,
                 gamma=2.0, label_smoothing=LABEL_SMOOTHING):  # uses config var (currently 0.1)
        super().__init__()
        self.lam1 = lam1; self.lam2 = lam2; self.lam3 = lam3
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=label_smoothing)
        self.supcon = SupConLoss(temperature=0.07)

    def focal(self, logits, y_soft):
        """Focal loss with soft targets."""
        p      = torch.softmax(logits, dim=1)
        log_p  = torch.log_softmax(logits, dim=1)
        weight = (1 - p) ** self.gamma
        return -(weight * log_p * y_soft).sum(dim=1).mean()

    def forward(self, logits, features, labels, y_soft=None):
        """
        logits   : (B, C) raw model output
        features : (B, D) bottleneck embedding for SupCon
        labels   : (B,)   hard integer labels
        y_soft   : (B, C) soft labels from Mixup (optional)
        """
        if y_soft is None:
            y_soft = F.one_hot(labels, logits.size(1)).float()
        # CE with soft targets via KL-div
        log_p   = F.log_softmax(logits, dim=1)
        ce_loss = -(y_soft * log_p).sum(dim=1).mean()
        # Focal with soft targets
        fc_loss = self.focal(logits, y_soft)
        # SupCon uses hard labels (Mixup labels don't affect contrastive)
        sc_loss = self.supcon(features, labels)
        # NaN guards -- prevent any single loss from poisoning training
        if torch.isnan(ce_loss): ce_loss = torch.tensor(0.0, device=ce_loss.device)
        if torch.isnan(fc_loss): fc_loss = torch.tensor(0.0, device=fc_loss.device)
        if torch.isnan(sc_loss): sc_loss = torch.tensor(0.0, device=sc_loss.device)
        return self.lam1 * ce_loss + self.lam2 * fc_loss + self.lam3 * sc_loss

print('HybridLoss ready (CE + Focal + SupCon + soft-target Mixup support).')


HybridLoss ready (CE + Focal + SupCon + soft-target Mixup support).


## Section 7 — Training Utilities

In [13]:

def cutmix_data(x, y, alpha=1.0):
    """CutMix augmentation — stronger than Mixup for medical images."""
    if alpha <= 0 or np.random.rand() > 0.5:
        return x, y, y, 1.0  # skip 50% of the time
    lam  = np.random.beta(alpha, alpha)
    B    = x.size(0)
    idx  = torch.randperm(B, device=x.device)
    W, H = x.size(3), x.size(2)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx    = np.random.randint(W)
    cy    = np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0); x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0); y2 = min(cy + cut_h // 2, H)
    x_mix = x.clone()
    x_mix[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return x_mix, y, y[idx], lam

# ====================================================================
# SECTION 7 -- UPGRADED TRAINING UTILITIES
# Adds: EMA, Cosine Warmup LR, Mixup, Label Smoothing, Grad Clip
# These are the primary drivers of smooth curves and ~96% accuracy.
# ====================================================================

# ── Exponential Moving Average (EMA) ─────────────────────────────────────
class EMA:
    """
    Maintains a shadow copy of model weights as an exponential moving average.
    EMA weights produce smoother, more stable predictions.
    Apply EMA weights at validation/test time for best accuracy.
    """
    def __init__(self, model, decay=0.9995):
        self.decay  = decay
        self.shadow = {k: v.clone().float() for k, v in model.state_dict().items()}

    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v.float()

    def apply(self, model):
        """Load EMA weights into model for inference."""
        model.load_state_dict({k: v.to(next(model.parameters()).device)
                               for k, v in self.shadow.items()})

    def state_dict(self):
        return self.shadow


# ── Cosine Annealing with Linear Warmup ──────────────────────────────────
class WarmupCosineScheduler:
    """
    Linear warmup for warmup_epochs, then cosine annealing to min_lr.
    Warmup prevents instability in early training (large pretrained models).
    Cosine decay produces smooth, stable convergence without oscillation.
    """
    def __init__(self, optimizer, warmup_epochs, total_epochs,
                 base_lr, min_lr=1e-6):
        self.optimizer     = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.base_lr       = base_lr
        self.min_lr        = min_lr
        self.current_epoch = 0

    def step(self):
        self.current_epoch += 1
        ep = self.current_epoch
        if ep <= self.warmup_epochs:
            lr = self.base_lr * ep / self.warmup_epochs
        else:
            import math
            progress = (ep - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (
                1.0 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
        return lr

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']


# ── Mixup augmentation ────────────────────────────────────────────────────
def mixup_batch(x, y, alpha=0.2, num_classes=4):
    """
    Mixup: interpolate pairs of training samples.
    Produces smoother decision boundaries and reduces overconfidence.
    Returns mixed inputs and soft targets as a float tensor.
    """
    import numpy as np
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch = x.size(0)
    idx   = torch.randperm(batch, device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    y_a   = F.one_hot(y, num_classes).float()
    y_b   = F.one_hot(y[idx], num_classes).float()
    y_mix = lam * y_a + (1 - lam) * y_b
    return x_mix, y_mix


# ── Upgraded train_one_epoch with Mixup + Grad Clip + EMA ────────────────
# Mixed-precision scaler (AMP) — ~2x faster training, no accuracy loss
_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def train_one_epoch(model, loader, optimizer, criterion,
                    ema=None, mixup_alpha=0.2, grad_clip=1.0,
                    scheduler=None):
    model.train()
    total_loss = 0.0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)

        # Mixup (training only)
        if mixup_alpha > 0:
            x_in, y_soft = mixup_batch(x, y, mixup_alpha, NUM_CLASSES)
        else:
            x_in  = x
            y_soft = F.one_hot(y, NUM_CLASSES).float()

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            feats  = model.forward_features(x_in)
            logits = model.classifier(feats)
            # Criterion accepts soft targets when mixup is active
            loss = criterion(logits, feats, y, y_soft=y_soft)

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad(); continue  # skip bad batch
        
        _scaler.scale(loss).backward()
        _scaler.unscale_(optimizer)
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        _scaler.step(optimizer)
        _scaler.update()

        # Update EMA after each step
        if ema is not None:
            ema.update(model)

        total_loss += loss.item() * x.size(0)

    # Step warmup/cosine scheduler per epoch
    if scheduler is not None and isinstance(scheduler, WarmupCosineScheduler):
        scheduler.step()

    return total_loss / len(loader.dataset)


# ── Eval epoch (unchanged but EMA-aware) ─────────────────────────────────
@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    probs_all, preds_all, lbls_all = [], [], []
    for x, y, _ in loader:
        logits = model(x.to(device))
        p = torch.softmax(logits, 1).cpu().numpy()
        probs_all.append(p)
        preds_all.append(p.argmax(1))
        lbls_all.append(y.numpy())
    return (np.concatenate(lbls_all),
            np.concatenate(preds_all),
            np.concatenate(probs_all))


# ── Metrics ────────────────────────────────────────────────────────────────
def all_metrics(lbls, preds, probs):
    m = {
        'accuracy':        accuracy_score(lbls, preds),
        'balanced_acc':    balanced_accuracy_score(lbls, preds),
        'f1_macro':        f1_score(lbls, preds, average='macro',    zero_division=0),
        'f1_weighted':     f1_score(lbls, preds, average='weighted', zero_division=0),
        'f2_macro':        fbeta_score(lbls, preds, beta=2, average='macro', zero_division=0),
        'precision_macro': precision_score(lbls, preds, average='macro', zero_division=0),
        'recall_macro':    recall_score(lbls, preds, average='macro',    zero_division=0),
        'mcc':             matthews_corrcoef(lbls, preds),
    }
    try:
        try:
            m['auc_macro']    = roc_auc_score(lbls, probs, multi_class='ovr', average='macro')
            m['auc_weighted'] = roc_auc_score(lbls, probs, multi_class='ovr', average='weighted')
        except Exception:
            # Fallback: model predicting single class (NaN loss collapse)
            m['auc_macro'] = m['auc_weighted'] = 0.5
    except Exception:
        m['auc_macro'] = m['auc_weighted'] = float('nan')
    return m


# ── Smooth curve plotting helper ──────────────────────────────────────────
def smooth(values, weight=0.85):
    """
    Exponential moving average smoothing for training curves.
    weight=0.85 gives visually smooth curves without losing trend information.
    """
    smoothed, last = [], values[0]
    for v in values:
        last = last * weight + v * (1 - weight)
        smoothed.append(last)
    return smoothed


def plot_training_curves(fold_histories, save_path=None):
    """
    Plot smooth training curves for all folds:
    - Loss (train)
    - Accuracy (val)
    - AUC-macro (val)
    - F1-macro (val)
    - Learning Rate
    """
    metrics = [
        ('loss',        'Training Loss',        '#e74c3c', True),
        ('accuracy',    'Val Accuracy',         '#3498db', False),
        ('auc_macro',   'Val AUC-macro',        '#2ecc71', False),
        ('f1_macro',    'Val F1-macro',         '#9b59b6', False),
    ]
    n_folds = len(fold_histories)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.ravel()

    colors = plt.cm.tab10(np.linspace(0, 1, n_folds))

    for ax_i, (metric, title, base_color, is_loss) in enumerate(metrics):
        ax = axes[ax_i]
        all_vals = []
        for fold_i, hist in enumerate(fold_histories):
            vals = [h.get(metric, float('nan')) for h in hist]
            eps  = [h['ep'] for h in hist]
            s_vals = smooth(vals, weight=0.85)
            # Raw (faint)
            ax.plot(eps, vals, alpha=0.18, color=colors[fold_i], linewidth=1)
            # Smoothed (bold)
            ax.plot(eps, s_vals, alpha=0.9,
                    color=colors[fold_i], linewidth=2.0,
                    label=f'Fold {fold_i+1}')
            all_vals.extend(vals)

        # Mean across folds
        max_ep = max(len(h) for h in fold_histories)
        mean_curve = []
        for ep in range(max_ep):
            ep_vals = [hist[ep].get(metric, float('nan'))
                       for hist in fold_histories if ep < len(hist)]
            mean_curve.append(np.nanmean(ep_vals))
        s_mean = smooth(mean_curve, weight=0.85)
        ax.plot(range(1, len(s_mean)+1), s_mean,
                color='black', linewidth=2.8, linestyle='--',
                label='Mean (smooth)', zorder=10)

        # Best value annotation
        best_val = min(all_vals) if is_loss else max(all_vals)
        ax.axhline(best_val, color='gray', linestyle=':', alpha=0.5,
                   linewidth=1.2)
        ax.text(0.98, best_val, f' Best: {best_val:.4f}',
                transform=ax.get_yaxis_transform(),
                ha='right', va='bottom', fontsize=8, color='gray')

        ax.set(title=title, xlabel='Epoch', ylabel=metric)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle('Training Curves -- Smooth (EMA) + Raw (faint)',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    out = save_path or (SAVE_DIR / 'figures' / 'training_curves_smooth.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[Curves] Smooth training curves saved -> {out}')


# ── Full training loop with EMA + Cosine Warmup (replaces train_es) ───────
def train_es(model, dl_tr, dl_va, optimizer, criterion,
             scheduler=None, max_ep=None, patience=None):
    """
    Upgraded training loop:
    - EMA weights tracked throughout
    - WarmupCosine scheduler used if scheduler is WarmupCosineScheduler
    - Mixup applied every batch
    - Gradient clipping every step
    - Best model selected by EMA-evaluated AUC
    - Smooth curves saved automatically
    """
    max_ep   = max_ep  or MAX_EPOCHS
    patience = patience or PATIENCE

    ema        = EMA(model, decay=EMA_DECAY)
    best_auc   = -1.0
    best_state = None
    best_ep    = 0
    bad        = 0
    hist       = []

    # Store original weights to restore after EMA eval
    for ep in range(1, max_ep + 1):
        # Progressive unfreezing: unfreeze all at epoch 10
        # [FIXED: Removed because it causes PyTorch AMP to crash]
        # if ep == 10:
        #     for param in model.parameters():
        #         param.requires_grad = True
        loss = train_one_epoch(
            model, dl_tr, optimizer, criterion,
            ema=ema, mixup_alpha=MIXUP_ALPHA, grad_clip=GRAD_CLIP,
            scheduler=scheduler if isinstance(scheduler, WarmupCosineScheduler) else None
        )

        # Evaluate with EMA weights
        _orig_state = copy.deepcopy(model.state_dict())
        ema.apply(model)
        lbls, preds, probs = eval_epoch(model, dl_va)
        model.load_state_dict(_orig_state)  # restore training weights

        m   = all_metrics(lbls, preds, probs)
        auc_val = m['auc_macro']
        hist.append({'ep': ep, 'loss': loss, **m})

        # Step ReduceLROnPlateau if used instead of cosine
        if scheduler is not None and isinstance(scheduler,
                torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(auc_val)

        cur_lr = optimizer.param_groups[0]['lr']

        print(f'Ep{ep:03d} loss={loss:.4f} AUC={auc_val:.4f} '
              f'Acc={m["accuracy"]:.4f} F1={m["f1_macro"]:.4f} '
              f'LR={cur_lr:.2e} '
              f'best={best_auc:.4f}@{best_ep} pat={bad}/{patience}')

        if auc_val > best_auc + 1e-4:
            best_auc   = auc_val
            best_ep    = ep
            bad        = 0
            # Save EMA weights as best state
            best_state = {k: v.cpu().clone() for k, v in ema.state_dict().items()}
        else:
            bad += 1

        if bad >= patience:
            print(f'Early stop @ ep{ep}  best AUC={best_auc:.4f}@ep{best_ep}')
            break

    # Load best EMA state into model
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return {
        'best_state': best_state,
        'best_ep':    best_ep,
        'best_auc':   best_auc,
        'hist':       hist,
    }


print('Training utilities ready (EMA + WarmupCosine + Mixup + GradClip + SmoothCurves).')

# ── Test Time Augmentation (TTA) ─────────────────────────────────────────────
# Runs inference with N augmented views and averages predictions.
# Adds ~1-2% accuracy at test time with zero extra training.
def tta_predict(model, df, n_aug=5, batch_size=BATCH_SIZE):
    """TTA: average predictions over n_aug augmented views."""
    import torch.nn.functional as F
    tta_transform = transforms.Compose([
        EnsureRGB(), CropBlackBorders(),
        SkullStripSimulation(blend_strength=0.1),
        BiasFieldCorrection(sigma=2.0),
        CLAHEEnhancement(clip_limit=1.5),
        GaussianDenoising(sigma=0.5),
        ROIGuidedCrop(cx_frac=0.50, cy_frac=0.53, roi_frac=0.48),
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.05, contrast=0.05),
        transforms.ToTensor(), ZScorePerImage(),
    ])
    model.eval()
    all_probs = []
    with torch.no_grad():
        for _ in range(n_aug):
            ds  = AlzheimerDataset(df, tta_transform)
            dl  = DataLoader(ds, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
            probs_run = []
            for x, _, _ in dl:
                x = x.to(device)
                out = model(x)
                probs_run.append(F.softmax(out, dim=1).cpu().numpy())
            all_probs.append(np.concatenate(probs_run, axis=0))
    return np.mean(all_probs, axis=0)  # average across n_aug views
print("TTA function ready.")


# ── Model Ensemble ────────────────────────────────────────────────────────────
# Average predictions from all fold models for maximum accuracy.
def ensemble_predict(models, df, use_tta=False, n_aug=3):
    """Ensemble: average softmax predictions from multiple models."""
    import torch.nn.functional as F
    all_model_probs = []
    for mdl in models:
        mdl.eval()
        if use_tta:
            probs = tta_predict(mdl, df, n_aug=n_aug)
        else:
            dl = make_loader(df, eval_transform, False)
            run_probs = []
            with torch.no_grad():
                for x, _, _ in dl:
                    out = mdl(x.to(device))
                    run_probs.append(F.softmax(out, dim=1).cpu().numpy())
            probs = np.concatenate(run_probs, axis=0)
        all_model_probs.append(probs)
    return np.mean(all_model_probs, axis=0)  # average across models
print("Ensemble function ready.")


Training utilities ready (EMA + WarmupCosine + Mixup + GradClip + SmoothCurves).
TTA function ready.
Ensemble function ready.


## Section 8 — Optuna Hyperparameter Tuning (TPE + MedianPruner)

In [14]:
# Dev split for Optuna
_pids  = df_train["patient_id"].unique()
_plab  = df_train.drop_duplicates("patient_id").set_index("patient_id").loc[_pids,"label"].values
_skf   = StratifiedKFold(5, shuffle=True, random_state=SEED)
_ti,_vi= next(_skf.split(_pids, _plab))
_df_otr = df_train[df_train["patient_id"].isin(_pids[_ti])].copy()
_df_ova = df_train[df_train["patient_id"].isin(_pids[_vi])].copy()
print(f"Optuna dev — train:{len(_df_otr)} val:{len(_df_ova)}")

def objective(trial):
    bd   = trial.suggest_categorical("bottleneck_dim",[256,512,768])
    fr   = trial.suggest_float("freeze_ratio",0.3,0.7)
    dr   = trial.suggest_float("dropout",0.3,0.7)   # ↑ wider range for ConvNeXt-Small
    lr   = trial.suggest_float("lr",5e-5,3e-4,log=True)   # narrowed range — known-good: 1.34e-4
    wd   = trial.suggest_float("weight_decay",1e-4,5e-3,log=True)  # narrowed: larger model needs L2 ≥ 1e-4
    ep   = trial.suggest_int("max_epochs", 15, 25)  # ↑ was [2,4] — critical fix: tune for real epoch count
    l1   = trial.suggest_float("lam1",0.2,0.6)
    l2   = trial.suggest_float("lam2",0.2,0.6)
    l3   = 1.0-l1-l2
    if l3 < 0.05: return float("inf")

    cw  = class_weights(_df_otr)
    mdl = DualChannelAD(NUM_CLASSES,bd,dr,fr).to(device)
    crit= HybridLoss(cw,l1,l2,l3)
    opt = torch.optim.AdamW(filter(lambda p:p.requires_grad, mdl.parameters()),lr=lr,weight_decay=wd)
    # Use WarmupCosine so Optuna tunes in the same LR landscape as real training
    sch = WarmupCosineScheduler(
        opt, warmup_epochs=max(2, ep//7), total_epochs=ep,  # ~14% warmup (was capped at 33%)
        base_lr=lr, min_lr=MIN_LR)
    dl_tr=make_loader(_df_otr,train_transform,True)
    dl_va=make_loader(_df_ova,eval_transform, False)
    es = train_es(mdl,dl_tr,dl_va,opt,crit,sch,max_ep=ep,patience=5)
    v  = es["best_auc"]
    del mdl; torch.cuda.empty_cache()
    return -v

study = optuna.create_study(direction="minimize",
                             sampler=TPESampler(seed=SEED),
                             pruner=MedianPruner(3,5))
study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=MAX_KAGGLE_SECONDS, show_progress_bar=True)
BP = study.best_params
print("\nBest params:", BP)
print(f"Best AUC   : {-study.best_value:.4f}")

# Save Optuna figure
fig,ax = plt.subplots(figsize=(10,4))
ax.plot([-t.value for t in study.trials], marker="o", lw=1.5, color="#3498db")
ax.set(xlabel="Trial",ylabel="Val AUC",title="Optuna — AUC per Trial"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(SAVE_DIR/"figures"/"optuna.png",dpi=150); plt.close()
pd.DataFrame([{**t.params,"value":t.value} for t in study.trials]).to_csv(
    SAVE_DIR/"results"/"optuna_trials.csv",index=False)
print("Optuna done.")

Optuna dev — train:3588 val:901


  0%|          | 0/10 [00:00<?, ?it/s]

Ep001 loss=0.8622 AUC=0.5557 Acc=0.4839 F1=0.2109 LR=2.20e-05 best=-1.0000@0 pat=0/5
Ep002 loss=0.6641 AUC=0.5840 Acc=0.4828 F1=0.2207 LR=4.41e-05 best=0.5557@1 pat=0/5
Ep003 loss=0.6283 AUC=0.6160 Acc=0.4806 F1=0.2264 LR=6.61e-05 best=0.5840@2 pat=0/5
Ep004 loss=0.6089 AUC=0.6497 Acc=0.4828 F1=0.2336 LR=6.58e-05 best=0.6160@3 pat=0/5
Ep005 loss=0.5840 AUC=0.6832 Acc=0.4883 F1=0.2381 LR=6.47e-05 best=0.6497@4 pat=0/5
Ep006 loss=0.5490 AUC=0.7101 Acc=0.4994 F1=0.3228 LR=6.29e-05 best=0.6832@5 pat=0/5
Ep007 loss=0.5562 AUC=0.7297 Acc=0.5072 F1=0.3757 LR=6.05e-05 best=0.7101@6 pat=0/5
Ep008 loss=0.5107 AUC=0.7461 Acc=0.5039 F1=0.3818 LR=5.74e-05 best=0.7297@7 pat=0/5
Ep009 loss=0.4938 AUC=0.7597 Acc=0.4961 F1=0.3792 LR=5.39e-05 best=0.7461@8 pat=0/5
Ep010 loss=0.4701 AUC=0.7713 Acc=0.4895 F1=0.3940 LR=4.98e-05 best=0.7597@9 pat=0/5
Ep011 loss=0.4689 AUC=0.7823 Acc=0.4850 F1=0.4028 LR=4.55e-05 best=0.7713@10 pat=0/5
Ep012 loss=0.4225 AUC=0.7917 Acc=0.4806 F1=0.4034 LR=4.08e-05 best=0.7823@

### Checkpoint Save -- After Section 8 (Optuna Hyperparams)
Saves best hyperparameters. **Run once after Optuna finishes.**
Next time load via Resume Loader to skip the tuning run.


In [15]:
# CHECKPOINT: Save best Optuna hyperparams after Section 8
import json as _json
from pathlib import Path
CKPT_DIR = Path('./outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

try:
    with open(CKPT_DIR / 'ckpt_best_params.json', 'w') as _f:
        _json.dump(BP if 'BP' in dir() else best_params, _f, indent=2)
    print('[CHECKPOINT SAVED] ckpt_best_params.json')
    for k, v in best_params.items():
        print(f'  {k}: {v}')
except NameError:
    print('[WARN] best_params not defined -- run Section 8 (Optuna) first')


[CHECKPOINT SAVED] ckpt_best_params.json
[WARN] best_params not defined -- run Section 8 (Optuna) first


## Section 9 — 5-Fold Patient-Level Cross-Validation

In [16]:
PARAMS = dict(
    bd  = BP.get("bottleneck_dim", 512),   
    fr  = BP.get("freeze_ratio",   0.5),
    dr  = BP.get("dropout",        0.4),
    lr  = BP.get("lr",             BASE_LR),  # default to BASE_LR=3e-4 if Optuna not run
    wd  = BP.get("weight_decay",   1e-4),
    ep  = BP.get("max_epochs",     MAX_EPOCHS),
    l1  = BP.get("lam1",           0.4),
    l2  = BP.get("lam2",           0.4),
    l3  = max(0.05, 1-BP.get("lam1",0.4)-BP.get("lam2",0.4)),
    pat = PATIENCE,  # use global PATIENCE=10 from Section 1
)


df_tv   = pd.concat([df_train,df_val],ignore_index=True)
u_pids  = df_tv["patient_id"].unique()
u_plab  = df_tv.drop_duplicates("patient_id").set_index("patient_id").loc[u_pids,"label"].values
o_skf   = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=SEED)
fold_mdls, fold_res = [], []

fold_histories = []  
for fold,(ti,vi) in enumerate(o_skf.split(u_pids,u_plab), 1):
    if time.time() - KAGGLE_START_TIME > MAX_KAGGLE_SECONDS:
        print('\n⚠️ Kaggle Time Limit Reached! Stopping CV early to safely save models.')
        break
    print(f"\n{'='*55}\n FOLD {fold}/{OUTER_FOLDS}\n{'='*55}")
    df_ftr = df_tv[df_tv["patient_id"].isin(u_pids[ti])]
    df_fva = df_tv[df_tv["patient_id"].isin(u_pids[vi])]
    cw  = class_weights(df_ftr)
    mdl = DualChannelAD(NUM_CLASSES, PARAMS["bd"], PARAMS["dr"], PARAMS["fr"]).to(device)
    crit= HybridLoss(cw,PARAMS["l1"],PARAMS["l2"],PARAMS["l3"])
    # Layerwise LR: backbone gets 0.1x LR, head gets full LR
    _backbone_params = [p for n, p in mdl.named_parameters()
                        if p.requires_grad and 'classifier' not in n]
    _head_params     = [p for n, p in mdl.named_parameters()
                        if p.requires_grad and 'classifier' in n]
    opt = torch.optim.AdamW([
        {'params': _backbone_params, 'lr': PARAMS['lr'] * 0.1},
        {'params': _head_params,     'lr': PARAMS['lr']},
    ], weight_decay=PARAMS['wd'])
    # WarmupCosine scheduler: smooth convergence, no oscillation
    sch = WarmupCosineScheduler(
        opt,
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=PARAMS['ep'],
        base_lr=PARAMS['lr'],
        min_lr=MIN_LR,
    )
    es  = train_es(mdl,make_loader(df_ftr,train_transform,True),
                   make_loader(df_fva,eval_transform,False),
                   opt,crit,sch,PARAMS["ep"],PARAMS["pat"])
    l,p,pr = eval_epoch(mdl, make_loader(df_fva,eval_transform,False))
    m = all_metrics(l,p,pr); m["fold"]=fold
    fold_res.append(m); fold_mdls.append(es["best_state"])
    fold_histories.append(es["hist"])
    torch.save(es["best_state"], SAVE_DIR/"models"/f"fold{fold}.pt")
    print(f"Fold{fold} AUC={m['auc_macro']:.4f} F1={m['f1_macro']:.4f}")
    del mdl; torch.cuda.empty_cache()

cv_df = pd.DataFrame(fold_res)
cv_df.to_csv(SAVE_DIR/"results"/"cv_results.csv",index=False)
print(f"\nMean AUC: {cv_df['auc_macro'].mean():.4f} ± {cv_df['auc_macro'].std():.4f}")
print(f"Mean F1 : {cv_df['f1_macro'].mean():.4f} ± {cv_df['f1_macro'].std():.4f}")

# Plot smooth training curves for all folds
plot_training_curves(fold_histories,
                     save_path=SAVE_DIR / 'figures' / 'training_curves_smooth.png')
print('[Section 9] Done. Smooth curves saved.')



 FOLD 1/5
Ep001 loss=0.9576 AUC=0.4633 Acc=0.0101 F1=0.0050 LR=1.32e-05 best=-1.0000@0 pat=0/8
Ep002 loss=0.7904 AUC=0.4779 Acc=0.0101 F1=0.0050 LR=2.64e-05 best=0.4633@1 pat=0/8
Ep003 loss=0.6997 AUC=0.5034 Acc=0.0147 F1=0.0204 LR=3.97e-05 best=0.4779@2 pat=0/8
Ep004 loss=0.6214 AUC=0.5429 Acc=0.0220 F1=0.0397 LR=5.29e-05 best=0.5034@3 pat=0/8
Ep005 loss=0.5761 AUC=0.5939 Acc=0.0549 F1=0.0824 LR=6.61e-05 best=0.5429@4 pat=0/8
Ep006 loss=0.5910 AUC=0.6476 Acc=0.1071 F1=0.0967 LR=6.57e-05 best=0.5939@5 pat=0/8
Ep007 loss=0.5594 AUC=0.6946 Acc=0.2299 F1=0.1861 LR=6.44e-05 best=0.6476@6 pat=0/8
Ep008 loss=0.4863 AUC=0.7286 Acc=0.3874 F1=0.3261 LR=6.22e-05 best=0.6946@7 pat=0/8
Ep009 loss=0.4878 AUC=0.7540 Acc=0.4542 F1=0.3613 LR=5.93e-05 best=0.7286@8 pat=0/8
Ep010 loss=0.4954 AUC=0.7720 Acc=0.4908 F1=0.3920 LR=5.56e-05 best=0.7540@9 pat=0/8
Ep011 loss=0.4888 AUC=0.7881 Acc=0.5101 F1=0.4698 LR=5.14e-05 best=0.7720@10 pat=0/8
Ep012 loss=0.4151 AUC=0.8011 Acc=0.5266 F1=0.4830 LR=4.66e-05 b

### Checkpoint Save -- After Section 9 (Centralized Model + Fold Data)
Saves trained centralized model and fold histories. **Run once after CV completes.**
Reload via Resume Loader to skip retraining next time.


In [17]:
# CHECKPOINT: Save centralized model + fold data after Section 9
import pickle, torch
from pathlib import Path
CKPT_DIR = Path('./outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

try:
    torch.save(cent_model.state_dict(), CKPT_DIR / 'ckpt_cent_model.pt')
    print('[CHECKPOINT SAVED] ckpt_cent_model.pt  '
          f'({sum(p.numel() for p in cent_model.parameters())/1e6:.1f}M params)')
except NameError:
    print('[WARN] cent_model not defined -- run Section 9 first')

try:
    _fold_ckpt = {
        'fold_histories': fold_histories if 'fold_histories' in dir() else [],
        'fold_mdls':      fold_mdls      if 'fold_mdls'      in dir() else [],
    }
    with open(CKPT_DIR / 'ckpt_fold_histories.pkl', 'wb') as _f:
        pickle.dump(_fold_ckpt, _f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'[CHECKPOINT SAVED] ckpt_fold_histories.pkl  '
          f'({len(_fold_ckpt["fold_histories"])} folds)')
except NameError as _e:
    print(f'[WARN] {_e} -- run Section 9 first')


[WARN] cent_model not defined -- run Section 9 first
[CHECKPOINT SAVED] ckpt_fold_histories.pkl  (5 folds)


### ✅ Part 1 Complete — Save All Checkpoints

All Optuna params and fold models are saved. Upload `outputs/` to a Kaggle Dataset before running Part 2.

In [18]:
# ====================================================================
# PART 1 FINAL SAVE — Upload outputs/ folder to Kaggle Dataset
# ====================================================================
import json, torch
from pathlib import Path

(SAVE_DIR / 'models').mkdir(parents=True, exist_ok=True)

# Save best Optuna hyperparams
with open(SAVE_DIR / 'models' / 'best_params.json', 'w') as f:
    json.dump(PARAMS, f, indent=2)
print(f"[Part1-Save] best_params.json → {SAVE_DIR / 'models' / 'best_params.json'}")

# Save every fold model
for fi, state in enumerate(fold_mdls):
    path = SAVE_DIR / 'models' / f'fold_{fi}_model.pt'
    torch.save(state, path)
    print(f"[Part1-Save] fold_{fi}_model.pt saved")

# Save best fold model separately (used by Parts 2 and 3 as cent_model)
# Find the fold with the highest AUC
best_fold_idx = max(range(len(fold_res)), key=lambda i: fold_res[i].get("auc_macro", 0))
print(f"[Part1-Save] Best fold: {best_fold_idx+1} (AUC={fold_res[best_fold_idx]["auc_macro"]:.4f})")
best_state = fold_mdls[best_fold_idx]
torch.save(best_state, SAVE_DIR / 'models' / 'best_fold_model.pt')
print(f"[Part1-Save] best_fold_model.pt saved (fold {best_fold_idx})")

# Save fold histories for Part 3 curves
import pickle
with open(SAVE_DIR / 'models' / 'fold_histories.pkl', 'wb') as f:
    pickle.dump(fold_histories, f)
print(f"[Part1-Save] fold_histories.pkl saved")

print()
print("=" * 60)
print("Part 1 COMPLETE. Next steps:")
print("  1. Go to Kaggle → Your notebook → Save Version")
print("  2. Datasets → New Dataset → upload outputs/ folder")
print("  3. Name it: alzheimer-part1-output")
print("  4. Open alzheimer_part2.ipynb and add that dataset as input")
print("=" * 60)


[Part1-Save] best_params.json → outputs/models/best_params.json
[Part1-Save] fold_0_model.pt saved
[Part1-Save] fold_1_model.pt saved
[Part1-Save] fold_2_model.pt saved
[Part1-Save] fold_3_model.pt saved
[Part1-Save] fold_4_model.pt saved
[Part1-Save] Best fold: 1 (AUC=0.8923)
[Part1-Save] best_fold_model.pt saved (fold 0)
[Part1-Save] fold_histories.pkl saved

Part 1 COMPLETE. Next steps:
  1. Go to Kaggle → Your notebook → Save Version
  2. Datasets → New Dataset → upload outputs/ folder
  3. Name it: alzheimer-part1-output
  4. Open alzheimer_part2.ipynb and add that dataset as input
